# Initializing kcat values for PAMparametrizer for the Yeast9 PAM

In this notebook, we give an overview of how an initial set of turnover number can be obtained. To enable this, we use multiple databases to map kcat values obtained from machine learning (from the [GotEnzymes](https://metabolicatlas.org/gotenzymes) database) to the proteins and reactions in the model using the gene-protein-reaction associations.

Before we start, we have to set up all the package and files required. Adjust the filenames and paths as desired. Importantly, make sure to **adjust the regular expression in the cell below** in order to find the gene names associated with the organism you want to model.

In [16]:
import pandas as pd
import os
from cobra.io import read_sbml_model
import matplotlib.pyplot as plt
import re
import numpy as np
import datetime

from Bio.KEGG import REST
import requests

if os.path.split(os.getcwd())[1]=='Scripts':
    os.chdir('..')

DEFAULT_PROT_MW = 39959.4825 #g/mol
DEFAULT_KCAT =  11.0 #/s median value from BRENDA
DEFAULT_KCAT_TRANSPORT = 100000000 #/s

METANETX_FILE_PATH = os.path.join('Data', 'Databases', 'reac_xref.tsv') #mapping rxn to EC number
mnx_chem_xref= pd.read_csv(os.path.join('Data', 'Databases',"chem_xref.tsv"), sep="\t",comment="#") #mapping chemicals
UNIPROT_FILE_PATH = os.path.join('Data', 'Databases','uniprotkb_scerevisiae_240903.xlsx')
RHEA2KEGG_FILE_PATH = os.path.join('Data', 'Databases','rhea2kegg_reaction.tsv')

GOTENZYMES_FILE_PATH = os.path.join('Data','Databases', 'GotEnzymes_sce_240903.json')

In [17]:
#output file path
# Create a datetime object for the current date
current_date = datetime.datetime.now()
# Format the date as yymmdd
formatted_date = current_date.strftime('%y%m%d')

OUTPUT_FILE_PATH = os.path.join('Data', f'proteinAllocationModel_yeast9_EnzymaticData_{formatted_date}.xlsx')
AE_OUTPUT_FILE_PATH = os.path.join('Data', f'{formatted_date}_sce_ActiveEnzymeInformation_GotEnzymes.xlsx')

#### Defining the regular expression for yeast locus tags
Explanation of the Regular Expression:

    Y: Matches the literal character "Y" (indicating it is a yeast ORF).
    [A-P]: Matches any uppercase letter between A and P, corresponding to the 16 yeast chromosomes (I to XVI).
    [LR]: Matches either "L" or "R", representing the left (L) or right (R) arm of the chromosome.
    [0-9]{3}: Matches exactly three digits (0-9), representing the location on the chromosome.
    [CW]: Matches either "C" (Crick strand) or "W" (Watson strand).

In [33]:
#You need to adjust this to find the geneid/locustag for your microbe
locustag_regex =r'(Y[A-P][LR][0-9]{3}[CW])'

## 0. Download information from different databases
1. **Metabolic model stoichiometries and reactions: [SysBioChalmers GitHub](https://github.com/SysBioChalmers)**
- In this example we will use the yeast9 model for [*Saccharomyces cerevisiae* CEN.PK 113-7D](https://github.com/SysBioChalmers/yeast-GEM/blob/main/model/yeast-GEM.xlsx)

2. **Turnover numbers: [GotEnzymes](https://metabolicatlas.org/gotenzymes)**
- Go to the [API of the Metabolic Atlas](https://metabolicatlas.org/api/v2/#/GotEnzymes/gotEnzymes)
- Use the `GET\gotenzymes\enzymes?` functionality, click `Try it out`
- Here we filled in `sce` (*S. cerevisiae* strain S288c (CEN.PK is very similar), KEGG organism code) for `organism` and `10000` for `pagination[pageSize]`. The rest was left empty
- Click execute and a json file with all the information will be downloaded
- Note: GotEnzymes gives the locus tag, ec number, reaction id and compound for each protein. The latter 2 are given as KEGG identifiers, which we subsequently have to map to BiGG identifiers in order to match with the model. Therefore, we need to use other databases

3. **Gene-Protein-Reaction relations: [UniprotKB](https://www.uniprot.org/)**
- Got to the [Uniprot knowledge base](https://www.uniprot.org/) and used advanced search to query for your organism. For *S.cerevisiae* we used [this](https://www.uniprot.org/uniprotkb?query=(taxonomy_id:4932)) webpage
- Click download, select `format`:`Excel` and select the following: 
    - `Uniprot Data -> Names & Taxonomy`: `Gene Names`
    - `Uniprot Data -> Sequences`: `length`, `mass`
    - `Uniprot Data -> Function`: `RheaID`
- Download the resulting Excel file
    
4. **Match KEGG reaction ids to Uniprot protein ids: [Rhea Database](https://www.rhea-db.org/)**
- Download the `rhea2kegg_reaction.tsv` file from the Cross-Reference section of the [download page of the rhea website](https://www.rhea-db.org/help/download)

5. **Match KEGG reaction ids to BIGG reaction ids: [MetaNetX](https://www.metanetx.org/)**
- Download the `chem_xref.tsv` and `reac_xref.tsv` flatfiles from the [MetaNetX namespace](https://www.metanetx.org/mnxdoc/mnxref.html)

## 1. Get information from the BiGG model

In the metabolic model itself is already quite some information stored. In order to obtain up-to-date information on the enzymes related to the reaction, we will need to obtain the kegg reaction ID, the reactants, the product and the gene-protein relation.

In [19]:
#load the model
model = read_sbml_model(os.path.join('Models', 'yeast9.xml'))

expand=False

#make a id mapping and create a mapping dataframe
id_mapper_df = pd.DataFrame(columns = ['rxn_id','bigg_id', 'kegg_id', 'rhea_id', 'mnx_id','Reactants','Products','EC', 'GPR'])
for rxn in model.reactions:
    to_append= [rxn.id]   

    #skip transport reactions and ATP maintenance requirement
    try:
        if 'ex' in rxn.annotation['bigg.reaction'] or 'EX' in rxn.annotation['bigg.reaction'] or rxn.annotation['bigg.reaction'] == 'ATPM':
            continue
        to_append.append(rxn.annotation['bigg.reaction'])
    except:
        to_append.append(np.nan)
    #not all reactions are annotated with a KEGG ID in the model
    try: 
        to_append.append(rxn.annotation['kegg.reaction'])
    except:
        to_append.append(np.nan)
    try: 
        to_append.append(rxn.annotation['rhea'])
    except:
        to_append.append(np.nan)
    try: 
        to_append.append(rxn.annotation['metanetx.reaction'])
    except:
        to_append.append(np.nan)
    try:
        to_append.append([reac.annotation['kegg.compound'] for reac in rxn.reactants])
    except:
        to_append.append([])
    try: 
        to_append.append([prod.annotation['kegg.compound'] for prod in rxn.products])
    except:
        to_append.append([])
    if 'ec-code' in rxn.annotation.keys():
        #if there are multiple enzymes make seperate lists
        if isinstance(rxn.annotation['ec-code'], list) and expand:
            info = to_append.copy()
            for i in range(len(rxn.annotation['ec-code'])):
                #make a new row for each enzyme by copying the previous information
                info = to_append.copy()
                ec= rxn.annotation['ec-code'][i]
                info.append(ec)
                #add gpr
                info.append(rxn.gene_reaction_rule)
                #add to df
                id_mapper_df.loc[len(id_mapper_df)] = info
        else:
            to_append.append(rxn.annotation['ec-code'])
            to_append.append(rxn.gene_reaction_rule)
            id_mapper_df.loc[len(id_mapper_df)] = to_append
    else:
        to_append.append(np.nan)
        #ignore entries without enzyme and gpr relation
        if rxn.gene_reaction_rule =='':
            continue
        else:
            to_append.append(rxn.gene_reaction_rule)
        id_mapper_df.loc[len(id_mapper_df)] = to_append
        
#remove entries without GRP relation or Ec number


print('Number of reactions in the mapping dataframe: ',len(id_mapper_df))
print('Number of reactions mapped to KEGG identifier: ', len(id_mapper_df.kegg_id.dropna()))
print('Number of reactions mapped to rhea identifier: ', len(id_mapper_df.rhea_id.dropna()))
print('Number of reactions mapped to MetaNetX identifier: ', len(id_mapper_df.mnx_id.dropna()))
id_mapper_df.head()

Number of reactions in the mapping dataframe:  2763
Number of reactions mapped to KEGG identifier:  866
Number of reactions mapped to rhea identifier:  14
Number of reactions mapped to MetaNetX identifier:  1234


,rxn_id,bigg_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR
0,r_0001,D_LACDcm,R00197,NaN,MNXR138960,"[C00256, C00125]","[C00126, C00022]","[1.1.2.4, 1.1.99.-]",(YDL174C and YEL039C) or (YDL174C and YJR048W)...
1,r_0002,D_LACDm,R00197,NaN,MNXR138960,"[C00256, C00125]","[C00126, C00022]","[1.1.2.4, 1.1.99.-]",(YDL178W and YEL039C) or (YDL178W and YJR048W)
2,r_0003,BTDD_RR,R02946,NaN,MNXR107844,"[C03044, C00003]","[C00810, C00080, C00004]",1.1.1.4,YAL060W
3,r_0004,L_LACD2cm,R00196,NaN,MNXR138959,"[C00186, C00125]","[C00126, C00022]",1.1.2.3,(YEL039C and YML054C) or (YJR048W and YML054C)
4,r_0005,NaN,R03118,NaN,MNXR143499,[C00029],"[C00965, C00080, C00015]",2.4.1.34,(YPR165W and YLR342W) or (YPR165W and YGR032W)...


## 2. Get missing kegg ids from MetaNetX

Normally, there are quite some reaction identifiers without KEGG annotation in the model. Therefore, we use the static information from MetaNetX to fill in some of the gaps

In [20]:
#load the mnx dataframe
mnx_reac_xref= pd.read_csv(METANETX_FILE_PATH, sep="\t",comment="#")
mnx_reac_xref.columns = ['ex_id', 'mnx_id', 'equation']
mnx_reac_xref

,ex_id,mnx_id,equation
0,bigg.reaction:CRBNTD,EMPTY,H2CO3 dissociation||1 biggM:h2co3@biggC:x = 1 ...
1,bigg.reaction:DNADRAIN,EMPTY,Dna sink transport reaction|| =
2,bigg.reaction:H2CO3D2,EMPTY,Carboxylic acid dissociation||1 biggM:h2co3@bi...
3,bigg.reaction:H2CO3D2m,EMPTY,"Carboxylic acid dissociation, mitochondrial||1..."
4,bigg.reaction:HMR_5409,EMPTY,HMR 5409||1 biggM:h@biggC:e + 1 biggM:hco3@big...
...,...,...,...
384796,biggR:R_GALNACT3g,MNXR99998,secondary/obsolete/fantasy identifier
384797,bigg.reaction:GALNACT4g,MNXR99999,Uridine diphosphoacetylgalactosamine-chondroit...
384798,bigg.reaction:R_GALNACT4g,MNXR99999,secondary/obsolete/fantasy identifier
384799,biggR:GALNACT4g,MNXR99999,Uridine diphosphoacetylgalactosamine-chondroit...


In [22]:
#parse the bigg and kegg ids to separate dataframes
mnx_reac_bigg = pd.DataFrame(columns=['mnx_id', 'bigg_id', 'bigg_substrates', 'bigg_products'])
mnx_reac_kegg = pd.DataFrame(columns=['mnx_id', 'kegg_id', 'substrates', 'products'])


for index, row in mnx_reac_xref.iterrows():
    #find bigg id using regex search
    bigg_pattern = re.search(r"bigg.reaction:(\w+)", row['ex_id'])
    if bigg_pattern and row['mnx_id']!= 'EMPTY': #check if there is a BiGG id and a mnx id
        index = len(mnx_reac_bigg)
        mnx_reac_bigg.loc[index, 'bigg_id'] = bigg_pattern.group(1)
        mnx_reac_bigg.loc[index, 'mnx_id'] = row['mnx_id']
        #if there is a equation available, extract the substrates and products into separate columns
        if 'biggM' in row['equation']:
            bigg_equation = row['equation'].split("||") #extract the equation part (last entry)
            equation_splitted = bigg_equation[-1].split('=') #split substrates and products
            
            #make a list of the substrates and products using a regex search
            substrates = [re.findall(" biggM:(.*)@biggC", str(compound)) for compound in equation_splitted[0].split('+')]
            products = [re.findall(" biggM:(.*)@biggC", str(compound)) for compound in equation_splitted[1].split('+')]
            
            #unlisting the individual compounds and skip empty entries
            mnx_reac_bigg.loc[index,'bigg_substrates'] = [comp[0] for comp in substrates if comp!= []]
            mnx_reac_bigg.loc[index,'bigg_products'] = [comp[0] for comp in products if comp!= []]
            
    #find kegg id using regex search
    kegg_pattern = re.search(r"kegg.reaction:(\w+)", row['ex_id'])
    if kegg_pattern:
        index = len(mnx_reac_kegg)
        mnx_reac_kegg.loc[index, 'kegg_id'] = kegg_pattern.group(1)
        mnx_reac_kegg.loc[index, 'mnx_id'] = row['mnx_id']
        #if there is a equation available, extract the substrates and products into separate columns
        if isinstance(row['equation'], str):
            kegg_equation = row['equation'].split("||") #extract the equation part (last entry)
            equation_splitted = kegg_equation[-1].split('=') #split substrates and products
            
            #make a list of the substrates and products using a regex search
            substrates = [re.findall(" keggC:(.*)@IN", str(compound)) for compound in equation_splitted[0].split('+')]
            products = [re.findall(" keggC:(.*)@IN", str(compound)) for compound in equation_splitted[1].split('+')]
            
            #unlisting the individual compounds and skip empty entries
            mnx_reac_kegg.loc[index,'substrates'] = [comp[0] for comp in substrates if comp!= []]
            mnx_reac_kegg.loc[index,'products'] = [comp[0] for comp in products if comp!= []]
            
print(mnx_reac_bigg)
print(mnx_reac_kegg)

          mnx_id      bigg_id      bigg_substrates        bigg_products
0         MNXR02       EX_h_e                   []                  [h]
1         MNXR02     R_EX_h_e                  NaN                  NaN
2         MNXR03     HMR_1095                  [h]                  [h]
3         MNXR03           Ht                  [h]                  [h]
4         MNXR03         Htcx                  [h]                  [h]
...          ...          ...                  ...                  ...
56321  MNXR99997  R_GALNACT2g                  NaN                  NaN
56322  MNXR99998    GALNACT3g  [h, cs_c_pre3, udp]  [cs_c_pre2, uacgam]
56323  MNXR99998  R_GALNACT3g                  NaN                  NaN
56324  MNXR99999    GALNACT4g  [h, cs_d_pre4, udp]  [cs_d_pre3, uacgam]
56325  MNXR99999  R_GALNACT4g                  NaN                  NaN

[56326 rows x 4 columns]
           mnx_id kegg_id                substrates                  products
0      MNXR100024  R00253  [C000

In [23]:
#there are roughly 5x more entries from BiGG than from KEGG
print('Number of entries from BiGG ', len(mnx_reac_bigg))
print('Number of entries from KEGG ', len(mnx_reac_kegg))

Number of entries from BiGG  56326
Number of entries from KEGG  11333


In [24]:
#match the bigg and kegg ids based on mnx id
mnx_reac_bigg_kegg = mnx_reac_bigg.merge(mnx_reac_kegg, on = 'mnx_id', how = 'inner')

print('Number of reaction bigg-mnx-kegg pairs ', len(mnx_reac_bigg_kegg))

Number of reaction bigg-mnx-kegg pairs  6712


In [25]:
kegg_matches_before = len(id_mapper_df.kegg_id.dropna())

#add the information to the id_mapper_df
for index, row in id_mapper_df.iterrows():
    #check if kegg dataframe is not known
    if row['kegg_id'] == '':
        #is there a match available from the metanetx data?
        bigg_match_kegg = mnx_reac_bigg_kegg['bigg_id'] == row['rxn_id']
        if sum(bigg_match_kegg)>0:
            #fill in the rows with the kegg info
            matches=mnx_reac_bigg_kegg[bigg_match_kegg]
            id_mapper_df.loc[index, 'kegg_id'] = matches['kegg_id'].iloc[0]
            id_mapper_df.loc[index, 'Reactants'] = matches['substrates'].iloc[0]
            id_mapper_df.loc[index, 'Products'] = matches['products'].iloc[0]
            
kegg_matches_after = len(id_mapper_df.kegg_id.dropna())
            
print('Number of reactions mapped to KEGG rxnid went from ', kegg_matches_before, ' to ', kegg_matches_after)

Number of reactions mapped to KEGG rxnid went from  866  to  866


### 2.1. Try to improve matching by matching bigg metabolites ids to kegg compound ids

In [9]:
#1. load chemical compound xref data
mnx_chem_xref.columns = ['ex_id', 'mnx_id', 'info']
mnx_chem_xref

,ex_id,mnx_id,info
0,mnx:BIOMASS,BIOMASS,BIOMASS
1,seed.compound:cpd11416,BIOMASS,Biomass
2,seedM:M_cpd11416,BIOMASS,secondary/obsolete/fantasy identifier
3,seedM:cpd11416,BIOMASS,Biomass
4,MNXM01,MNXM01,PMF||Translocated proton that acccounts for th...
...,...,...,...
2996504,sabiork.compound:40,WATER,H2O||Water
2996505,sabiorkM:40,WATER,H2O||Water
2996506,seed.compound:cpd00001,WATER,H2O||H20||H3O+||HO-||Hydroxide ion||OH||OH-||W...
2996507,seedM:M_cpd00001,WATER,secondary/obsolete/fantasy identifier


In [10]:
# parse the bigg and kegg ids to separate dataframes
mnx_chem_bigg = pd.DataFrame(columns=['mnx_id', 'bigg_id'])
mnx_chem_kegg = pd.DataFrame(columns=['mnx_id', 'kegg_id'])


for index, row in mnx_chem_xref.iterrows():
    #find bigg id using regex search
    bigg_pattern = re.search(r"bigg.metabolite:(\w+)", row['ex_id'])
    if bigg_pattern and row['mnx_id']!= 'EMPTY': #check if there is a BiGG id and a mnx id
        index = len(mnx_chem_bigg)
        mnx_chem_bigg.loc[index, 'bigg_id'] = bigg_pattern.group(1)
        mnx_chem_bigg.loc[index, 'mnx_id'] = row['mnx_id']
        
    #find kegg id using regex search
    kegg_pattern = re.search(r"kegg.compound:(\w+)", row['ex_id'])
    if kegg_pattern:
        index = len(mnx_chem_kegg)
        mnx_chem_kegg.loc[index, 'kegg_id'] = kegg_pattern.group(1)
        mnx_chem_kegg.loc[index, 'mnx_id'] = row['mnx_id']
            
print(mnx_chem_bigg)
print(mnx_chem_kegg)

         mnx_id      bigg_id
0        MNXM02          oh1
1         MNXM1            h
2        MNXM10         nadh
3       MNXM100         grdp
4     MNXM10053  mercplaccys
...         ...          ...
9082    MNXM988        34hpl
9083    MNXM990     3htmelys
9084    MNXM992       4mptnl
9085   MNXM9931      35cdamp
9086      WATER          h2o

[9087 rows x 2 columns]
          mnx_id kegg_id
0         MNXM02  C01328
1          MNXM1  C00080
2         MNXM10  C00004
3        MNXM100  C00341
4      MNXM10006  C16496
...          ...     ...
18806   MNXM9986  C06057
18807   MNXM9987  C05145
18808   MNXM9989  C16720
18809   MNXM9990  C16266
18810      WATER  C00001

[18811 rows x 2 columns]


In [11]:
# mnx_reac_bigg.to_csv(os.join(BASEDIR, 'Results', 'mnx_reac_bigg.csv'))
kegg_matches = id_mapper_df[id_mapper_df['Products'].str.len()>0]
kegg_matches_before = len(kegg_matches[kegg_matches['Reactants'].str.len()>0])

In [12]:
# # match compound information to the id_mapper_df
# kegg_matches = id_mapper_df[id_mapper_df['Products'].str.len()>0]
# kegg_matches_before = len(kegg_matches[kegg_matches['Reactants'].str.len()>0])

# #add the information to the id_mapper_df
# for index, row in id_mapper_df.iterrows():
#     #check if kegg dataframe is not known
#     if row['Products'] == [] and row['Reactants'] == []:
#         #is there a match available from the metanetx data?
#         bigg_match = mnx_reac_bigg['bigg_id'] == row['rxn_id']
#         if sum(bigg_match)>0:
#             bigg_row = mnx_reac_bigg[bigg_match]
#             #fill in the rows with the kegg info
#             kegg_products = []
#             for product in bigg_row['bigg_products'].iloc[0]:
#                 mnx_chem_id= mnx_chem_bigg[mnx_chem_bigg['bigg_id'] == product]
#                 kegg_chem_id =mnx_chem_kegg[mnx_chem_kegg['mnx_id'] == mnx_chem_id]['kegg_id']
#                 kegg_products += [kegg_chem_id.iloc[0]]
#             kegg_substrates = []
                
#             for substrate in bigg_row['bigg_substrates'].iloc[0]:
#                 mnx_chem_id= mnx_chem_bigg[mnx_chem_bigg['bigg_id'] == substrate]
#                 kegg_chem_id =mnx_chem_kegg[mnx_chem_kegg['mnx_id'] == mnx_chem_id]['kegg_id']
#                 kegg_substrates += [kegg_chem_id.iloc[0]]
                
            
#             id_mapper_df.loc[index, 'Reactants'] = kegg_substrates
#             id_mapper_df.loc[index, 'Products'] = kegg_products
            
# kegg_matches = id_mapper_df[id_mapper_df['Products'].str.len()>0]
# kegg_matches_after = len(kegg_matches[kegg_matches['Reactants'].str.len()>0])
            
# print('Number of reactions mapped to KEGG compound ids went from ', kegg_matches_before, ' to ', kegg_matches_after)

## 4. Get all the protein information

Besides information about the reactions, we need the relation to the proteins in order to establish a protein allocation model. We do this using the gene-protein-reaction association obtained from the BIGG model. The genes is the model can be mapped to the Uniprot accessions, which can be mapped to the KEGG reactions using the Rhea database.

### 4.1 Parse the BIGG gene-protein-reaction associations

In [26]:
#You need to adjust this to find the geneid/locustag for your microbe
#locustag_regex =r'\b([b|s]\d{4})\b'
def extract_b_numbers(text):
    return re.findall(locustag_regex, text)

id_mapper_df['b_number'] = id_mapper_df['GPR'].apply(extract_b_numbers)
id_mapper_df = id_mapper_df.explode('b_number', ignore_index=True)
id_mapper_df

,rxn_id,bigg_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR,b_number
0,r_0001,D_LACDcm,R00197,NaN,MNXR138960,"[C00256, C00125]","[C00126, C00022]","[1.1.2.4, 1.1.99.-]",(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YDL174C
1,r_0001,D_LACDcm,R00197,NaN,MNXR138960,"[C00256, C00125]","[C00126, C00022]","[1.1.2.4, 1.1.99.-]",(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YEL039C
2,r_0001,D_LACDcm,R00197,NaN,MNXR138960,"[C00256, C00125]","[C00126, C00022]","[1.1.2.4, 1.1.99.-]",(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YDL174C
3,r_0001,D_LACDcm,R00197,NaN,MNXR138960,"[C00256, C00125]","[C00126, C00022]","[1.1.2.4, 1.1.99.-]",(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YJR048W
4,r_0001,D_LACDcm,R00197,NaN,MNXR138960,"[C00256, C00125]","[C00126, C00022]","[1.1.2.4, 1.1.99.-]",(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YEL039C
...,...,...,...,...,...,...,...,...,...,...
4304,r_4774,NaN,R00139,NaN,MNXR106405,"[C00002, C00363]","[C00008, C00459]",2.7.4.6,YKL067W,YKL067W
4305,r_4775,NaN,R00139,NaN,MNXR106405,"[C00002, C00035]","[C00008, C00044]",2.7.4.6,YKL067W,YKL067W
4306,r_4776,NaN,R00139,NaN,MNXR106405,"[C00002, C00104]","[C00008, C00081]",2.7.4.6,YKL067W,YKL067W
4307,r_4777,NaN,R00139,NaN,MNXR106405,"[C00002, C01344]","[C00008, C01345]",2.7.4.6,YKL067W,YKL067W


### 4.2 Parse the Uniprot data for merging

In [34]:
# load the uniprot information
uniprot_df = pd.read_excel(UNIPROT_FILE_PATH)
#get the gene id from the gene names
uniprot_df['b_number'] = uniprot_df['Gene Names'].str.extract(locustag_regex)
uniprot_df = uniprot_df.drop('Gene Names', axis=1)
uniprot_df

,Entry,Length,Mass,Rhea ID,b_number
0,A0A0B7P3V8,1104,127189,RHEA:22508 RHEA:22508,YPL060C
1,A6ZM04,859,97628,RHEA:13065,NaN
2,A6ZU07,897,101717,RHEA:17989 RHEA:46608,NaN
3,A6ZXH8,997,115452,RHEA:38571 RHEA:38663 RHEA:38895 RHEA:67920,NaN
4,A6ZY89,1588,174807,RHEA:21968 RHEA:21096 RHEA:17737 RHEA:13121 RH...,NaN
...,...,...,...,...,...
48029,V9H1E1,26,3084,NaN,NaN
48030,V9H1G1,175,20671,NaN,NaN
48031,V9H1G4,26,3077,NaN,NaN
48032,V9H1L5,152,17301,NaN,NaN


### 4.3 Match Uniprot to BIGG data

In [35]:
id_mapper_with_protein = pd.merge(id_mapper_df, uniprot_df, on='b_number', how='left')

id_mapper_with_protein

,rxn_id,bigg_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR,b_number,Entry,Length,Mass,Rhea ID
0,r_0001,D_LACDcm,R00197,NaN,MNXR138960,"[C00256, C00125]","[C00126, C00022]","[1.1.2.4, 1.1.99.-]",(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YDL174C,P32891,587,65293,RHEA:13521
1,r_0001,D_LACDcm,R00197,NaN,MNXR138960,"[C00256, C00125]","[C00126, C00022]","[1.1.2.4, 1.1.99.-]",(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YEL039C,P00045,113,12532,NaN
2,r_0001,D_LACDcm,R00197,NaN,MNXR138960,"[C00256, C00125]","[C00126, C00022]","[1.1.2.4, 1.1.99.-]",(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YDL174C,P32891,587,65293,RHEA:13521
3,r_0001,D_LACDcm,R00197,NaN,MNXR138960,"[C00256, C00125]","[C00126, C00022]","[1.1.2.4, 1.1.99.-]",(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YJR048W,P00044,109,12182,NaN
4,r_0001,D_LACDcm,R00197,NaN,MNXR138960,"[C00256, C00125]","[C00126, C00022]","[1.1.2.4, 1.1.99.-]",(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YEL039C,P00045,113,12532,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2452399,r_4774,NaN,R00139,NaN,MNXR106405,"[C00002, C00363]","[C00008, C00459]",2.7.4.6,YKL067W,YKL067W,P36010,153,17167,RHEA:44640 RHEA:18113
2452400,r_4775,NaN,R00139,NaN,MNXR106405,"[C00002, C00035]","[C00008, C00044]",2.7.4.6,YKL067W,YKL067W,P36010,153,17167,RHEA:44640 RHEA:18113
2452401,r_4776,NaN,R00139,NaN,MNXR106405,"[C00002, C00104]","[C00008, C00081]",2.7.4.6,YKL067W,YKL067W,P36010,153,17167,RHEA:44640 RHEA:18113
2452402,r_4777,NaN,R00139,NaN,MNXR106405,"[C00002, C01344]","[C00008, C01345]",2.7.4.6,YKL067W,YKL067W,P36010,153,17167,RHEA:44640 RHEA:18113


In [36]:
#get the RHEA ids from the rhea id column and create an individual row for each id
# Split the strings into individual 'RHEA' entries and explode them into separate rows
id_mapper_with_protein['rhea_id_up'] = id_mapper_with_protein['Rhea ID'].str.split(' ')
id_mapper_with_protein = id_mapper_with_protein.explode('rhea_id_up', ignore_index=True)

# Extract the 'RHEA' IDs using a regular expression
id_mapper_with_protein['rhea_id_up'] = id_mapper_with_protein['rhea_id_up'].str.extract(r'RHEA:(\d+)')

# Reset the index to get a clean DataFrame
id_mapper_with_protein_parsed = id_mapper_with_protein.reset_index(drop=True).drop(['Rhea ID','rhea_id', 'mnx_id'], axis =1)

id_mapper_with_protein_parsed

,rxn_id,bigg_id,kegg_id,Reactants,Products,EC,GPR,b_number,Entry,Length,Mass,rhea_id_up
0,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]","[1.1.2.4, 1.1.99.-]",(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YDL174C,P32891,587,65293,13521
1,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]","[1.1.2.4, 1.1.99.-]",(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YEL039C,P00045,113,12532,NaN
2,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]","[1.1.2.4, 1.1.99.-]",(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YDL174C,P32891,587,65293,13521
3,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]","[1.1.2.4, 1.1.99.-]",(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YJR048W,P00044,109,12182,NaN
4,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]","[1.1.2.4, 1.1.99.-]",(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YEL039C,P00045,113,12532,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2539931,r_4776,NaN,R00139,"[C00002, C00104]","[C00008, C00081]",2.7.4.6,YKL067W,YKL067W,P36010,153,17167,18113
2539932,r_4777,NaN,R00139,"[C00002, C01344]","[C00008, C01345]",2.7.4.6,YKL067W,YKL067W,P36010,153,17167,44640
2539933,r_4777,NaN,R00139,"[C00002, C01344]","[C00008, C01345]",2.7.4.6,YKL067W,YKL067W,P36010,153,17167,18113
2539934,r_4778,NaN,R00139,"[C00002, C01346]","[C00008, C00460]",2.7.4.6,YKL067W,YKL067W,P36010,153,17167,44640


### 4.3 Use the Rhea ID from uniprot to find the right KEGG ids

In [37]:
rhea2kegg_df = pd.read_csv(RHEA2KEGG_FILE_PATH, sep ='\t')
rhea2kegg_df = rhea2kegg_df.drop(['DIRECTION', 'MASTER_ID'], axis=1)
rhea2kegg_dict = rhea2kegg_df.to_dict()

In [38]:
nmbr_kegg_mapped = 0
for i, row in id_mapper_with_protein_parsed.iterrows():
    if isinstance(row.kegg_id, float):
        if row.rhea_id_up in rhea2kegg_dict.keys():
            id_mapper_with_protein_parsed.kegg_id.iloc[i] = rhea2kegg_dict[row.rhea_id_up]
            nmbr_kegg_mapped += 1
print('additional number of KEGG ids which were mapped were: ', nmbr_kegg_mapped)

additional number of KEGG ids which were mapped were:  0


### 5. Match the BiGG model reactiona and enzymes to the kcats from GotEnzymes
Find first all enzymes for a reaction, then for each enzyme determine if the kcats from GotEnzymes are for a forward or reverse reaction

In [39]:
id_mapper_final = id_mapper_with_protein_parsed
id_mapper_final = id_mapper_final.drop(['rhea_id_up'], axis = 1).rename({'Entry':'uniprot_id'}, axis=1)
id_mapper_final = id_mapper_final.explode('kegg_id', ignore_index=True)
id_mapper_final = id_mapper_final.explode('EC', ignore_index=True)

# id_mapper_final = id_mapper_final.drop_duplicates(subset = ['kegg_id'])
id_mapper_final.head()

,rxn_id,bigg_id,kegg_id,Reactants,Products,EC,GPR,b_number,uniprot_id,Length,Mass
0,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]",1.1.2.4,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YDL174C,P32891,587,65293
1,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]",1.1.99.-,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YDL174C,P32891,587,65293
2,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]",1.1.2.4,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YEL039C,P00045,113,12532
3,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]",1.1.99.-,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YEL039C,P00045,113,12532
4,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]",1.1.2.4,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YDL174C,P32891,587,65293


In [40]:
#get the json file from GotEnzymes and parse into workable format
#read in the json file obtained from gotEnzymes (for 'eco')
eco_enzymes_json = pd.read_json(GOTENZYMES_FILE_PATH)
#convert to a readible format
eco_enzymes = eco_enzymes_json.enzymes.to_list()
eco_enzymes_df = pd.DataFrame(eco_enzymes)

#ensure there is only one ec number per row
eco_enzymes_df['ec_number'] = eco_enzymes_df['ec_number'].str.split(pat=';')
eco_enzymes_df = eco_enzymes_df.explode('ec_number', ignore_index=True)
eco_enzymes_df = eco_enzymes_df.dropna(axis=0, subset=['ec_number'])
eco_enzymes_df = eco_enzymes_df.drop(['organism', 'domain'], axis = 1)
eco_enzymes_df.head()

,gene,reaction_id,ec_number,compound,kcat_values
0,YAL012W,R00782,4.4.1.1,C00283,2.2484
1,YAL012W,R00782,4.4.1.13,C00283,2.2484
2,YAL012W,R00782,4.4.1.28,C00283,2.2484
3,YAL012W,R04930,4.4.1.1,C00109,1.1646
4,YAL012W,R04930,4.4.1.1,C00014,4.2309


In [41]:
# match BiGG and GotEnzymes based on gene id
eco_enzymes_merged = id_mapper_final.merge(eco_enzymes_df, 
                              left_on =['b_number', 'kegg_id'], 
                              right_on = ['gene', 'reaction_id'],
                             how = 'left').drop(['gene', 'reaction_id'], axis=1).rename({'b_number':'gene'}, axis=1)
eco_enzymes_merged

,rxn_id,bigg_id,kegg_id,Reactants,Products,EC,GPR,gene,uniprot_id,Length,Mass,ec_number,compound,kcat_values
0,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]",1.1.2.4,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YDL174C,P32891,587,65293,1.1.2.4,C00256,51.3780
1,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]",1.1.2.4,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YDL174C,P32891,587,65293,1.1.2.4,C00022,35.7262
2,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]",1.1.99.-,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YDL174C,P32891,587,65293,1.1.2.4,C00256,51.3780
3,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]",1.1.99.-,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YDL174C,P32891,587,65293,1.1.2.4,C00022,35.7262
4,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]",1.1.2.4,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YEL039C,P00045,113,12532,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3004431,r_4777,NaN,R00139,"[C00002, C01344]","[C00008, C01345]",2.7.4.6,YKL067W,YKL067W,P36010,153,17167,2.7.4.6,C11038,13.6041
3004432,r_4778,NaN,R00139,"[C00002, C01346]","[C00008, C00460]",2.7.4.6,YKL067W,YKL067W,P36010,153,17167,2.7.4.6,C11039,13.4934
3004433,r_4778,NaN,R00139,"[C00002, C01346]","[C00008, C00460]",2.7.4.6,YKL067W,YKL067W,P36010,153,17167,2.7.4.6,C11038,13.6041
3004434,r_4778,NaN,R00139,"[C00002, C01346]","[C00008, C00460]",2.7.4.6,YKL067W,YKL067W,P36010,153,17167,2.7.4.6,C11039,13.4934


### 6. Extract directionalities and gapfill
Reactions which are associated with an enzyme, but not with a kcat from GotEnzymes will be assignes
a default kcat and if required a default molmass.

In [42]:
# Get directionalities and fill in the gaps
eco_enzymes_merged['direction'] = 'f'
for index, row in eco_enzymes_merged.iterrows():
    #there is a kegg id and kcat associated to this reaction
    if not isinstance(row.kcat_values, float):
        if row.compound in row.Products:
            eco_enzymes_merged.direction.iloc[index] = 'b'
        elif not row.compound in row.Reactants:
            print(f'Compound {row["compound"]} is not associated to reaction {row["kegg_id"]}')
            continue
    #there is no kcat associated to this reaction, try to use EC number to get a kcat value
    else: 
        got_enzyme_info = eco_enzymes_df[eco_enzymes_df['ec_number']==row.EC]
        for i, info in got_enzyme_info.iterrows():
            #is there any enzyme association if we look at the metabolites in the reaction and the EC number?
            if info.compound in row.Products:
                direction = 'b'
            elif info.compound in row.Reactants:
                direction = 'f'
            else:
                continue
            #change the kcat if it is not there already or create a new enzyme association
            if np.isnan(row.kcat_values):
                eco_enzymes_merged.direction.iloc[index] = 'b'
                eco_enzymes_merged.kcat_values.iloc[index] = info.kcat_values
            else:
                eco_enzymes_merged.loc[len(eco_enzymes_merged)] = row.to_list()[:-2]+[info.kcat_values, 'b']
        #add default information for unmappable proteins
        if np.isnan(row.kcat_values) & (isinstance(row.uniprot_id, str)) & (row.GPR!= 's0001'):
            eco_enzymes_merged.direction.iloc[index] = 'f'
            eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT
        if np.isnan(row.Mass) & (len(row.GPR)>2) & (row.GPR!= 's0001'):
            eco_enzymes_merged.Mass.iloc[index] = DEFAULT_PROT_MW
        
        
eco_enzymes_mapped = eco_enzymes_merged.drop(['ec_number', 'compound'], axis=1).rename({'Mass':'molMass'}, axis =1)
eco_enzymes_mapped

/tmp/ipykernel_137260/3802666525.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'b'
/tmp/ipykernel_137260/3802666525.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.kcat_values.iloc[index] = info.kcat_values
/tmp/ipykernel_137260/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_137260/380

/tmp/ipykernel_137260/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_137260/3802666525.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT
/tmp/ipykernel_137260/3802666525.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'b'
/tmp/ipykernel_137260/3802666

/tmp/ipykernel_137260/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_137260/3802666525.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT
/tmp/ipykernel_137260/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_137260/3802666

/tmp/ipykernel_137260/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_137260/3802666525.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT
/tmp/ipykernel_137260/3802666525.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'b'
/tmp/ipykernel_137260/3802666

/tmp/ipykernel_137260/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_137260/3802666525.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT
/tmp/ipykernel_137260/3802666525.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'b'
/tmp/ipykernel_137260/3802666

...

/tmp/ipykernel_137260/3802666525.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'b'
/tmp/ipykernel_137260/3802666525.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.kcat_values.iloc[index] = info.kcat_values
/tmp/ipykernel_137260/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_137260/380

/tmp/ipykernel_137260/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_137260/3802666525.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT
/tmp/ipykernel_137260/3802666525.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'b'
/tmp/ipykernel_137260/3802666

/tmp/ipykernel_137260/3802666525.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'b'
/tmp/ipykernel_137260/3802666525.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.kcat_values.iloc[index] = info.kcat_values
/tmp/ipykernel_137260/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_137260/380

/tmp/ipykernel_137260/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_137260/3802666525.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT
/tmp/ipykernel_137260/3802666525.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'b'
/tmp/ipykernel_137260/3802666

,rxn_id,bigg_id,kegg_id,Reactants,Products,EC,GPR,gene,uniprot_id,Length,molMass,kcat_values,direction
0,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]",1.1.2.4,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YDL174C,P32891,587,65293,51.3780,f
1,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]",1.1.2.4,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YDL174C,P32891,587,65293,35.7262,f
2,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]",1.1.99.-,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YDL174C,P32891,587,65293,51.3780,f
3,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]",1.1.99.-,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YDL174C,P32891,587,65293,35.7262,f
4,r_0001,D_LACDcm,R00197,"[C00256, C00125]","[C00126, C00022]",1.1.2.4,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,YEL039C,P00045,113,12532,11.0000,f
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3046922,r_4778,NaN,R00139,"[C00002, C01346]","[C00008, C00460]",2.7.4.6,YKL067W,YKL067W,P36010,153,17167,15.8160,b
3046923,r_4778,NaN,R00139,"[C00002, C01346]","[C00008, C00460]",2.7.4.6,YKL067W,YKL067W,P36010,153,17167,15.3622,b
3046924,r_4778,NaN,R00139,"[C00002, C01346]","[C00008, C00460]",2.7.4.6,YKL067W,YKL067W,P36010,153,17167,15.3622,b
3046925,r_4778,NaN,R00139,"[C00002, C01346]","[C00008, C00460]",2.7.4.6,YKL067W,YKL067W,P36010,153,17167,15.3622,b


In [43]:
#save the dataframe
eco_enzymes_mapped = eco_enzymes_mapped.drop_duplicates(subset = ['rxn_id', 'gene', 'direction'],
                                                       ignore_index=True)
eco_enzymes_mapped.to_excel(AE_OUTPUT_FILE_PATH)

### 7. Save the dataframe to the proper format for building PAMs

In [49]:
# Get the information about the enzyme sectors
translational_protein_df = pd.DataFrame({'Parameter': ['id_list', 'tps_0', 'tps_mu', 'mol_mass'],
                                        'Value': ['r_2111', 0.08,0.35, 1388.488*1e3],
                                        'Unit': ['', 'g_ribosome/g_protein', 'g_ribosome/g_protein/h', '𝑔/𝑚𝑜𝑙'],
                                        'Description': ['biomass formation', 'Elsemman et al 2021 (Fig S10)',
                                                       'Elsemman et al 2021 (Fig S10)',
                                                        'Elsemman et al 2021 (Fig S10)']})

PAM_DATA_FILE_PATH = os.path.join('Data', 'proteinAllocationModel_iML1515_EnzymaticData_new.xls')

unused_enzymes_df = pd.read_excel(PAM_DATA_FILE_PATH,
                                 sheet_name = 'ExcessEnzymes') #need to parse it later

In [52]:
# Save it to a new excel file
with pd.ExcelWriter(OUTPUT_FILE_PATH) as writer:
    eco_enzymes_mapped.to_excel(writer, sheet_name='ActiveEnzymes', index=False)
    translational_protein_df.to_excel(writer, sheet_name='Translational', index =False)
    unused_enzymes_df.to_excel(writer, sheet_name = 'UnusedEnzyme', index=False)